# 👟 Gaussian Splatting: uCO3D shoe → gsplat

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Pipeline: COLMAP dataset (made on the Mac) → train with gsplat `simple_trainer.py` (v1.5.3) → `.ply` splat saved to Google Drive.

## 1. Check the GPU

In [ ]:
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Load the COLMAP dataset from Google Drive
Upload `shoe_colmap.zip` (the zipped `ws/undistorted` folder) to the top level of **My Drive** first.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/shoe_colmap.zip"   # change if you put it elsewhere
!rm -rf /content/data && mkdir -p /content/data
!unzip -q "$ZIP_PATH" -d /content/data

# find the folder that contains images/ + sparse/0/
import glob, os
hits = [os.path.dirname(os.path.dirname(p)) for p in glob.glob("/content/data/**/sparse/0", recursive=True)
        if "__MACOSX" not in p]
DATA_DIR = hits[0]
print("DATA_DIR =", DATA_DIR)
print("images:", len(os.listdir(f"{DATA_DIR}/images")), "| sparse/0:", os.listdir(f"{DATA_DIR}/sparse/0"))

## 3. Install gsplat + the example trainer
Uses the gsplat **v1.5.3** release so the pip package and the trainer script match. Colab's own torch/numpy are kept.

In [ ]:
!pip install -q gsplat==1.5.3 ninja
!rm -rf /content/gsplat
!git clone -q --branch v1.5.3 https://github.com/nerfstudio-project/gsplat.git /content/gsplat

# trainer requirements, minus the numpy<2 pin (would break Colab) and optional deps we don't use
!grep -v -E "numpy|fused-bilagrid|fused-ssim" /content/gsplat/examples/requirements.txt > /content/reqs.txt
!pip install -q -r /content/reqs.txt
# fused-ssim compiles CUDA code against Colab's torch, so it needs --no-build-isolation
!pip install -q --no-build-isolation "git+https://github.com/rahul-goel/fused-ssim@328dc9836f513d00c4b5bc38fe30478b4435cbb5"

## 4. Compile gsplat's CUDA kernels (one time, ~5–10 min)
It looks frozen while compiling. That's normal.

In [ ]:
!python -c "from gsplat.cuda._backend import _C; print('gsplat CUDA kernels ready')"

## 5. Train
Start with **7,000 steps** (~10–15 min on a T4) to check everything works. Then set `STEPS = 30000` for full quality (~45–60 min).

In [ ]:
STEPS = 7000
RESULT_DIR = f"/content/results/shoe_{STEPS}"

%cd /content/gsplat/examples
!python simple_trainer.py default \
    --data_dir "$DATA_DIR" \
    --data_factor 1 \
    --result_dir "$RESULT_DIR" \
    --max_steps $STEPS \
    --save_ply \
    --disable_viewer

## 6. Check the results: metrics + a held-out render vs. ground truth

In [ ]:
import json, glob
from IPython.display import Image, display

for f in sorted(glob.glob(f"{RESULT_DIR}/stats/val_step*.json")):
    print(os.path.basename(f), json.load(open(f)))

renders = sorted(glob.glob(f"{RESULT_DIR}/renders/val_step*_0000.png"))
if renders:
    print("left = ground truth, right = your splat")
    display(Image(renders[-1], width=900))
print(glob.glob(f"{RESULT_DIR}/ply/*.ply"))

## 7. Save to Google Drive (Colab deletes `/content` when the session ends)
Open the `.ply` in a web splat viewer like **SuperSplat** (drag and drop) to fly around it.

In [ ]:
!mkdir -p /content/drive/MyDrive/splats
!cp -r "$RESULT_DIR" /content/drive/MyDrive/splats/
print("Saved to My Drive/splats/" + os.path.basename(RESULT_DIR))